# 03. Multivariate Dependence, DCC-GARCH and Forbes-Rigobon Adjustment

The univariate GARCH notebook told us whether each market became more volatile after COVID-19. This notebook now moves to the second dimension of the project: **co-movement**. If the structure of risk changed, it should not only appear in each asset separately, but also in the way markets move together.

We therefore study:
- rolling 52-week correlations across key pairs;
- a bivariate DCC-GARCH model for the central SPX / US 10Y relation;
- the Forbes-Rigobon adjustment to correct raw correlation comparisons when volatility changes sharply across samples.

## 1. Setup

We reuse the same cleaned daily series built in notebook 01. This guarantees that any change in co-movement comes from the data rather than from a different alignment rule or transformation choice.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from project2_multivariate_utils import (
    KEY_PAIRS,
    PAIR_LABELS,
    ROLLING_WINDOW,
    load_multivariate_samples,
    rolling_correlation_table,
    plot_rolling_correlations,
    summarize_rolling_correlations,
    fit_bivariate_dcc,
    plot_spx_ust_dcc,
    summarize_dcc_path,
    build_forbes_rigobon_table,
    save_multivariate_outputs,
)
from project2_data_utils import ensure_output_dirs, save_figure

sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = "{:.4f}".format
ensure_output_dirs()

## 2. Load the aligned daily sample

The analysis again relies on the exact pre/post split used in the first two notebooks. This matters because rolling correlations and DCC estimates are very sensitive to missing observations and date alignment.

In [ ]:
aligned_returns, pre_covid, post_covid = load_multivariate_samples()

print(f"Full aligned sample: {aligned_returns.shape}")
print(f"Pre-COVID sample:    {pre_covid.shape}")
print(f"Post-COVID sample:   {post_covid.shape}")

## 3. Rolling 52-week correlations

A rolling correlation is the simplest way to visualize whether dependence is stable or regime-dependent. We use a **252-trading-day window**, which is the natural daily approximation to one year. This lets us see whether the COVID period marks a durable shift in the sign, level or instability of key cross-asset links.

We focus on four pairs that are economically central:
- S&P 500 vs US 10Y yield change,
- S&P 500 vs US HY Bonds,
- S&P 500 vs Oil futures,
- Gold vs US 10Y yield change.

In [ ]:
rolling_table = rolling_correlation_table(aligned_returns, KEY_PAIRS, window=ROLLING_WINDOW)
rolling_summary = summarize_rolling_correlations(rolling_table)
rolling_summary

In [ ]:
rolling_figure = plot_rolling_correlations(rolling_table)
save_figure(rolling_figure, "03_rolling_correlations.png")
plt.show()

## 4. Why the SPX / US 10Y pair deserves special treatment

Among all pairs, the SPX / US 10Y relation is the most informative for the project. It captures the interaction between risky assets and the benchmark sovereign yield curve. In calm times, this link often reflects growth expectations. In stress periods, it may flip sign or become unstable as markets move into a flight-to-quality regime.

A rolling correlation is helpful, but it remains a reduced descriptive object. To go one step further, we estimate a **bivariate DCC-GARCH** model for this pair.

In [ ]:
spx_ust_pair = aligned_returns[["date", "sp500", "ust10y_yield"]].dropna().copy()
dcc_result = fit_bivariate_dcc(spx_ust_pair)
dcc_summary = pd.DataFrame([{
    "dcc_alpha": dcc_result["alpha"],
    "dcc_beta": dcc_result["beta"],
    "dcc_persistence": dcc_result["persistence"],
    "optimization_success": dcc_result["success"],
}])
dcc_summary

## 5. Dynamic conditional correlation path

The DCC path gives a filtered measure of time-varying dependence after controlling for univariate volatility clustering in each series. Economically, this is important because a raw correlation can move simply because each market becomes more volatile, not necessarily because the transmission mechanism itself changes.

In [ ]:
dcc_path = dcc_result["dcc_path"]
dcc_path.head()

In [ ]:
dcc_figure = plot_spx_ust_dcc(dcc_path)
save_figure(dcc_figure, "03_spx_ust_dcc_path.png")
plt.show()

In [ ]:
dcc_path_summary = summarize_dcc_path(dcc_path)
dcc_path_summary

## 6. Forbes-Rigobon adjustment

A raw correlation comparison can exaggerate contagion when one market becomes mechanically more volatile across samples. Forbes and Rigobon proposed a correction for this bias. We therefore apply their adjustment to the **post-COVID SPX / US 10Y correlation**, using the increase in S&P 500 variance as the scaling term.

The interpretation is straightforward:
- if the adjusted post-COVID correlation remains far from the pre-COVID correlation, the change is less likely to be just a volatility artifact;
- if the adjustment largely removes the gap, then the apparent correlation shift is partly mechanical.

In [ ]:
forbes_rigobon_table = build_forbes_rigobon_table(pre_covid, post_covid)
forbes_rigobon_table

## 7. Save multivariate outputs

We save the rolling-correlation summary, the DCC summary, the DCC path and the Forbes-Rigobon comparison so that the next notebooks can refer to these results directly.

In [ ]:
save_multivariate_outputs(rolling_summary, dcc_summary, forbes_rigobon_table, dcc_path)
print("Saved multivariate outputs under outputs/project2/")

## 8. Main takeaways from notebook 03

This notebook tells us whether the co-movement structure of risk changed after COVID-19. The rolling correlations provide a first descriptive answer, the DCC model offers a volatility-adjusted dynamic view for the key SPX / US 10Y pair, and the Forbes-Rigobon adjustment helps us judge whether the observed change is genuine or partly driven by the post-COVID jump in market variance.